# 4. Synthetic scenes (future work, not used in the current scope)
Pastes isolated characters into multi-character scenes with YOLO labels, for the future multi-character detector (notebook 05). Source images are split into train/val/test BEFORE pasting, so no source image appears in more than one split.

In [1]:
import os, sys
from pathlib import Path

# run everything from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))  # so `backend.ml.preprocessing` can be imported
print("project root:", ROOT)

project root: D:\Portfolio Projects\MODI Script Identification


In [2]:
# ---- settings ----
SRC = "data/processed/offline_subset"
OUT = "data/processed/yolo_scenes"
N_TRAIN, N_VAL, N_TEST = 4000, 500, 500
IMG_SIZE = 640
MIN_CHARS, MAX_CHARS = 2, 6
PER_CLASS = 300     # same as the other notebooks
SEED = 42

## Shared split
Same function in notebooks 02 to 05, so every method is trained and tested on the same source images when `DATA`, `PER_CLASS` and `SEED` match.

In [3]:
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# The crop function lives in the backend because the API must preprocess exactly like training.
from backend.ml.preprocessing import crop_square

EXTS = {".png", ".jpg", ".jpeg", ".bmp"}


def select_and_split(root, per_class=300, seed=42):
    """Returns (classes, paths, labels, {"train": idx, "val": idx, "test": idx}); 70/15/15, stratified.
    Deterministic: the same root, per_class and seed always give the same split."""
    root = Path(root)
    classes = sorted(d.name for d in root.iterdir() if d.is_dir())
    assert classes, f"no class folders found in {root}"
    rng = np.random.RandomState(seed)
    paths, labels = [], []
    for ci, c in enumerate(classes):
        files = sorted(f for f in (root / c).iterdir() if f.suffix.lower() in EXTS)
        if per_class and len(files) > per_class:
            keep = np.sort(rng.choice(len(files), per_class, replace=False))
            files = [files[i] for i in keep]
        paths.extend(files)
        labels.extend([ci] * len(files))
    labels = np.array(labels)
    idx = np.arange(len(paths))
    tr, tmp = train_test_split(idx, test_size=0.3, stratify=labels, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.5, stratify=labels[tmp], random_state=seed)
    return classes, paths, labels, {"train": tr, "val": va, "test": te}


def load_crops(paths, size):
    """(N, size, size) uint8: tight-cropped, padded square, dark ink on white (same as the API)."""
    out = np.full((len(paths), size, size), 255, np.uint8)
    bad = 0
    for i, p in enumerate(paths):
        g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if g is None:
            bad += 1
            continue
        out[i] = crop_square(g, size)
    if bad:
        print(f"warning: {bad} unreadable images were left blank")
    return out

In [4]:
import json, random

## Helpers

In [5]:
def load_char(path, side, rng):
    """Ink-strength mask (0-255), tight-cropped, rotated slightly."""
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        return None
    if g.mean() < 127:
        g = 255 - g
    mask = 255 - g
    ys, xs = np.where(mask > 40)
    if len(xs) == 0:
        return None
    mask = mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h0, w0 = mask.shape
    scale = side / max(h0, w0)
    w, h = max(8, int(w0 * scale)), max(8, int(h0 * scale))
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_AREA)

    ang = rng.uniform(-8, 8)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos) + 2, int(h * cos + w * sin) + 2
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    mask = cv2.warpAffine(mask, M, (nw, nh))
    ys, xs = np.where(mask > 40)
    if len(xs) == 0:
        return None
    return mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def make_scene(pool, size, rng, nrng, nmin, nmax):
    chars, used = [], 0
    for _ in range(rng.randint(nmin, nmax)):
        cls, path = rng.choice(pool)
        m = load_char(path, rng.randint(48, 110), rng)
        if m is None:
            continue
        gap = rng.randint(8, 40)
        if used + m.shape[1] + gap > size - 40:
            break
        chars.append((cls, m, gap))
        used += m.shape[1] + gap
    if not chars:
        return None

    canvas = np.full((size, size), rng.randint(190, 255), np.float32)
    canvas += nrng.normal(0, rng.uniform(2, 10), canvas.shape)
    canvas *= np.linspace(rng.uniform(0.85, 1.0), 1.0, size, dtype=np.float32)[None, :]

    ink = rng.randint(0, 70)
    x = rng.randint(10, max(10, size - 10 - used))
    base_y = rng.randint(60, size - 60)
    labels = []
    for cls, m, gap in chars:
        h, w = m.shape
        if x + w > size - 5:
            break
        y = int(base_y - h / 2 + rng.randint(-10, 10))
        y = min(max(y, 5), size - h - 5)
        a = m.astype(np.float32) / 255.0
        region = canvas[y:y + h, x:x + w]
        canvas[y:y + h, x:x + w] = region * (1 - a) + ink * a
        labels.append((cls, (x + w / 2) / size, (y + h / 2) / size, w / size, h / size))
        x += w + gap
    if not labels:
        return None

    k = rng.choice([1, 3, 5])
    img = np.clip(canvas, 0, 255).astype(np.uint8)
    if k > 1:
        img = cv2.GaussianBlur(img, (k, k), 0)
    return img, labels

## Generate

In [6]:
rng = random.Random(SEED)
nrng = np.random.default_rng(SEED)
out = Path(OUT).resolve()
classes, paths, labels, split_idx = select_and_split(SRC, PER_CLASS, SEED)
pools = {s: [(int(labels[i]), paths[i]) for i in idx] for s, idx in split_idx.items()}

for split, n in {"train": N_TRAIN, "val": N_VAL, "test": N_TEST}.items():
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)
    assert pools[split], f"no source images for split '{split}'"
    made = 0
    while made < n:
        res = make_scene(pools[split], IMG_SIZE, rng, nrng, MIN_CHARS, MAX_CHARS)
        if res is None:
            continue
        img, scene_labels = res
        stem = f"{split}_{made:06d}"
        cv2.imwrite(str(out / "images" / split / f"{stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
        with open(out / "labels" / split / f"{stem}.txt", "w") as f:
            for cls, cx, cy, w, h in scene_labels:
                f.write(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
        made += 1
    print(f"{split}: {made} scenes")

yaml = [f"path: {out}", "train: images/train", "val: images/val", "test: images/test", "names:"]
yaml += [f"  {i}: {json.dumps(c)}" for i, c in enumerate(classes)]
(out / "data.yaml").write_text("\n".join(yaml) + "\n")
print(f"wrote {out / 'data.yaml'} ({len(classes)} classes)")

train: 4000 scenes
val: 500 scenes
test: 500 scenes
wrote D:\Portfolio Projects\MODI Script Identification\data\processed\yolo_scenes\data.yaml (57 classes)
